# Irrigation Need Prediction — Kaggle Playground S6E4
**Task:** Multiclass Classification (Low / Medium / High)  
**Metric:** Balanced Accuracy

This notebook walks through loading the data, preprocessing, comparing models, cross-validating, and generating the final submission.

In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully")

All libraries imported successfully


## 1. Load the Data
We load train, test, and the sample submission. The test IDs are saved separately — we need them to build the final submission file.

In [4]:
train      = pd.read_csv("train.csv")
test       = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Sample sub  : {sample_sub.shape}")
print()
print(train.head(3))
print()
print("Target distribution:")
print(train["Irrigation_Need"].value_counts())

Train shape : (630000, 21)
Test shape  : (270000, 20)
Sample sub  : (270000, 2)

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   
2            6.05  ...       Rice        Vegetative  Kharif       Sprinkler   

  Water_Source Field_Area_hectare  Mulching_Used Previous_Irrigation_mm  \
0    Rainwater   

## 2. Save IDs and Drop Unnecessary Columns
The uid=0(root) gid=0(root) groups=0(root) column is just a row identifier — not useful for learning. We pull the test IDs out before dropping so we can use them in the submission.

In [5]:
test_ids = test["id"].copy()

train = train.drop(columns=["id"])
test  = test.drop(columns=["id"])

print(f"Train: {train.shape}  |  Test: {test.shape}")

Train: (630000, 20)  |  Test: (270000, 19)


## 3. Separate Features and Target

In [6]:
TARGET = "Irrigation_Need"

X = train.drop(columns=[TARGET])
y = train[TARGET]

print(f"X shape : {X.shape}")
print(f"Classes : {sorted(y.unique())}")

X shape : (630000, 19)
Classes : ['High', 'Low', 'Medium']


## 4. Encode the Target
 converts the class labels (Low / Medium / High) into integers. We keep the encoder so we can reverse the mapping on the test predictions.

In [7]:
le    = LabelEncoder()
y_enc = le.fit_transform(y)

print("Label encoding:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls:>6}  ->  {idx}")

Label encoding:
    High  ->  0
     Low  ->  1
  Medium  ->  2


## 5. Identify Column Types
Categorical columns need . Numerical columns just need the median imputer (no missing values here, but good practice).

In [8]:
# Detect categorical columns robustly across pandas versions
cat_cols = [c for c in X.columns if str(X[c].dtype) in ("object", "string", "str")]
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

print(f"Categorical ({len(cat_cols)}): {cat_cols}")
print(f"Numerical   ({len(num_cols)}): {num_cols}")

Categorical (8): ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical   (11): ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


## 6. Build the Preprocessing Pipeline
 applies different transformations to categorical and numerical columns in one clean step — no manual encoding, no leakage risk.

In [9]:
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

print("Preprocessor ready")

Preprocessor ready


## 7. Train / Validation Split
Stratified 80/20 split — stratification matters here because the classes are imbalanced (Low >> Medium >> High).

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc,
    test_size=0.20,
    random_state=42,
    stratify=y_enc
)

print(f"Training rows   : {X_train.shape[0]:,}")
print(f"Validation rows : {X_val.shape[0]:,}")

Training rows   : 504,000
Validation rows : 126,000


## 8. Define Models
We compare four classifiers ranging from simple to powerful:
- **Decision Tree** — interpretable baseline
- **Naive Bayes** — fast probabilistic model
- **Logistic Regression** — linear baseline
- **Random Forest** — ensemble, usually strong on tabular data

In [11]:
models = {
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Naive Bayes":         GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42, n_jobs=-1),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}
print("Models defined:", list(models.keys()))

Models defined: ['Decision Tree', 'Naive Bayes', 'Logistic Regression', 'Random Forest']


## 9. Train and Evaluate All Models
Each model is wrapped in its own end-to-end pipeline (preprocessor + classifier). We score everything on the validation set using Balanced Accuracy.

In [12]:
results   = {}
pipelines = {}

print("=" * 55)
print(f"{'Model':<25} {'Balanced Accuracy':>18}")
print("=" * 55)

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier",   clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    score  = balanced_accuracy_score(y_val, y_pred)

    results[name]   = score
    pipelines[name] = pipe
    print(f"{name:<25} {score:>18.4f}")

print("=" * 55)

Model                      Balanced Accuracy
Decision Tree                         0.9494
Naive Bayes                           0.7794
Logistic Regression                   0.6654
Random Forest                         0.9579


## 10. Select the Best Model
We pick whichever model achieved the highest Balanced Accuracy on the validation set.

In [13]:
best_model_name  = max(results, key=results.get)
best_model_score = results[best_model_name]

print(f"Best model : {best_model_name}")
print(f"Val score  : {best_model_score:.4f}  (Balanced Accuracy)")

Best model : Random Forest
Val score  : 0.9579  (Balanced Accuracy)


## 11. K-Fold Cross Validation
Running 5-fold stratified CV on the best model gives a more reliable performance estimate than a single train/val split.

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier",   models[best_model_name])
    ]),
    X, y_enc,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1
)

print(f"5-Fold CV for {best_model_name}:")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"  Mean  : {cv_scores.mean():.4f}")
print(f"  Std   : {cv_scores.std():.4f}")

5-Fold CV for Random Forest:
  Fold 1: 0.9547
  Fold 2: 0.9570
  Fold 3: 0.9568
  Fold 4: 0.9568
  Fold 5: 0.9531
  Mean  : 0.9557
  Std   : 0.0015


## 12. Retrain on Full Training Data
Now we drop the validation holdout and retrain on 100% of the labelled data, giving the model the most possible information before predicting on the test set.

In [15]:
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   models[best_model_name])
])

final_pipeline.fit(X, y_enc)
print(f"{best_model_name} retrained on {len(X):,} rows")

Random Forest retrained on 630,000 rows


## 13. Predict on Test Set

In [17]:
test_preds_enc = final_pipeline.predict(test)
test_preds     = le.inverse_transform(test_preds_enc)

print(f"Predictions: {len(test_preds):,}")
print("Prediction distribution:")
unique, counts = np.unique(test_preds, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {cls:<10}: {cnt:>7,}  ({cnt / len(test_preds) * 100:.1f}%)")

Predictions: 270,000
Prediction distribution:
  High      :   8,404  (3.1%)
  Low       : 159,920  (59.2%)
  Medium    : 101,676  (37.7%)


## 14. Create submission.csv
The file must exactly match : two columns — uid=0(root) gid=0(root) groups=0(root) and .

In [19]:
submission = pd.DataFrame({
    "id":               test_ids.values,
    "Irrigation_Need":  test_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")
print()
print("First 10 rows:")
print(submission.head(10).to_string(index=False))

submission.csv saved!

First 10 rows:
    id Irrigation_Need
630000             Low
630001             Low
630002             Low
630003             Low
630004             Low
630005          Medium
630006             Low
630007          Medium
630008          Medium
630009             Low


## 15. Final Summary

In [23]:
print("=" * 60)
print("                  FINAL SUMMARY")
print("=" * 60)
print()
print("Model Comparison (Balanced Accuracy on validation set):")
for name, score in sorted(results.items(), key=lambda kv: kv[1], reverse=True):
    marker = "  <-- BEST" if name == best_model_name else ""
    print(f"  {name:<25}  {score:.4f}{marker}")

print()
print(f"Best Model      : {best_model_name}")
print(f"Val Score       : {best_model_score:.4f}")
print(f"CV Mean Score   : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print()
print(f"Submission rows : {len(submission):,}")
print(f"Columns         : {list(submission.columns)}")
print()
print("Prediction distribution:")
print(submission["Irrigation_Need"].value_counts().to_string())
print()

                  FINAL SUMMARY

Model Comparison (Balanced Accuracy on validation set):
  Random Forest              0.9579  <-- BEST
  Decision Tree              0.9494
  Naive Bayes                0.7794
  Logistic Regression        0.6654

Best Model      : Random Forest
Val Score       : 0.9579
CV Mean Score   : 0.9557 +/- 0.0015

Submission rows : 270,000
Columns         : ['id', 'Irrigation_Need']

Prediction distribution:
Irrigation_Need
Low       159920
Medium    101676
High        8404

